# Cell selection demo

Train the CNN–ViT with MSE, MSE + CE, and SPO+, then compare marker error and top-10% selection utility.

Download `results/colab_extended_demo.zip` and upload it when prompted. The sample contains 1,201 cells from complete training fields, with a separate 80/20 field split. The five-epoch runs illustrate the workflow; the full-study results are in `results/tables/`.


In [ ]:
# Install only dependencies that are absent from the current runtime.
import importlib.util, subprocess, sys
packages = {'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy',
            'sklearn': 'scikit-learn', 'torch': 'torch>=2.2', 'matplotlib': 'matplotlib'}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import hashlib, json, os, stat, tempfile, zipfile
from pathlib import Path, PurePosixPath
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.nn import functional as F
print('PyTorch', torch.__version__, '| NumPy', np.__version__)

In [ ]:
# CELLSELECT_DEMO_ZIP can point to the bundle during a local automated run.
archive_path = Path(os.environ.get('CELLSELECT_DEMO_ZIP', 'colab_extended_demo.zip'))
if not archive_path.is_file():
    try:
        from google.colab import files
    except ImportError:
        archive_path = Path(input('Path to colab_extended_demo.zip: ').strip()).expanduser()
    else:
        uploaded = files.upload()
        candidates = [name for name in uploaded if name.endswith('.zip')]
        if len(candidates) != 1:
            raise ValueError('Upload exactly one colab_extended_demo.zip file.')
        archive_path = Path(candidates[0])
bundle = Path(tempfile.mkdtemp(prefix='cell-selection-demo-')).resolve()
with zipfile.ZipFile(archive_path) as archive:
    if sum(item.file_size for item in archive.infolist()) > 512 * 1024**2:
        raise ValueError('Unexpectedly large demonstration bundle.')
    for item in archive.infolist():
        name = PurePosixPath(item.filename.replace(chr(92), '/'))
        if (name.is_absolute() or '..' in name.parts or ':' in str(name)
                or stat.S_ISLNK(item.external_attr >> 16)):
            raise ValueError('Unsafe archive member.')
    archive.extractall(bundle)
metadata = json.loads((bundle / 'metadata.json').read_text())
for name, expected in metadata['files_sha256'].items():
    if hashlib.sha256((bundle / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f'Bundle checksum mismatch: {name}')
assert metadata['source_partition'] == 'train'
assert metadata['main_validation_or_test_cells_included'] == 0
sys.path.insert(0, str(bundle / 'src'))
for name in list(sys.modules):
    if name == 'cellselect' or name.startswith('cellselect.'):
        del sys.modules[name]
print(f"Loaded {metadata['n_cells']:,} training-partition cells from {metadata['n_fields']} fields.")


In [ ]:
from cellselect.model import build_model, parameter_counts
from cellselect.losses import topk_spo_plus
from cellselect.training import seed_everything, predict
from cellselect.metrics import prediction_metrics, selection_metrics
with np.load(bundle / 'sample.npz', allow_pickle=False) as sample:
    images, measured = sample['images'].copy(), sample['marker'].copy()
    classes, ids = sample['class_id'].copy(), sample['global_index'].copy()
    groups = sample['split_group'].copy()
order = np.random.default_rng(metadata['demo_split_seed']).permutation(np.unique(groups))
train = np.flatnonzero(np.isin(groups, order[:int(0.8 * len(order))]))
holdout = np.flatnonzero(~np.isin(groups, order[:int(0.8 * len(order))]))
assert len(train) and len(holdout) and set(classes[train]) == {0, 1, 2}
assert not set(groups[train]) & set(groups[holdout])
# Every normalization statistic is fitted on the demo training rows.
train_pixels = images[train].astype(np.float64)
image_mean = train_pixels.mean(axis=(0, 2, 3))
image_std = train_pixels.std(axis=(0, 2, 3))
target_mean, target_std = measured[train].mean(), measured[train].std()
assert (image_std > 0).all() and target_std > 0
del train_pixels
device = torch.device(os.environ.get('CELLSELECT_DEMO_DEVICE',
                                    'cuda' if torch.cuda.is_available() else 'cpu'))
torch.set_num_threads(min(int(os.environ.get('CELLSELECT_DEMO_THREADS', '4')), os.cpu_count() or 1))
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
mean = torch.tensor(image_mean, dtype=torch.float32, device=device)[None, :, None, None]
std = torch.tensor(image_std, dtype=torch.float32, device=device)[None, :, None, None]
x_train, x_holdout = torch.from_numpy(images[train]), torch.from_numpy(images[holdout])
y_train = torch.tensor((measured[train] - target_mean) / target_std, dtype=torch.float32)
c_train = torch.tensor(classes[train], dtype=torch.long)
print(f'{len(train)} train / {len(holdout)} holdout cells; device: {device}')
print('Trainable model parameters:', parameter_counts(build_model(seed=17))['trainable'])

In [ ]:
# Fixed seed, initialization, ordering, epochs and optimizer for all three objectives.
seed, epochs, batch_size = 17, 5, 128
predictions, rows = {}, []
for objective, weight, name in [('mse', 0.0, 'MSE'), ('mse', 1.0, 'MSE + CE'), ('spo_plus', 0.0, 'SPO+')]:
    seed_everything(seed)
    model = build_model(seed=seed).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    generator = torch.Generator().manual_seed(seed)
    for epoch in range(epochs):
        model.train()
        permutation = torch.randperm(len(train), generator=generator)
        loss_sum = 0.0
        for index in permutation.split(batch_size):
            x = (x_train[index].to(device) - mean) / std
            y, c = y_train[index].to(device), c_train[index].to(device)
            optimizer.zero_grad(set_to_none=True)
            output = model(x)
            assert output['marker'].shape == y.shape
            primary = (topk_spo_plus(output['marker'], y, fraction=0.1) if objective == 'spo_plus'
                       else F.mse_loss(output['marker'], y))
            loss = primary + weight * F.cross_entropy(output['logits'], c)
            if not torch.isfinite(loss):
                raise FloatingPointError('Nonfinite demonstration loss.')
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(index)
        print(f'{name}: epoch {epoch + 1}/{epochs}, training loss {loss_sum / len(train):.3f}')
    logits, marker = predict(model, x_holdout, mean, std, device, batch_size, amp=False)
    score = marker.astype(np.float64) * target_std + target_mean
    predictions[name] = score
    metrics = prediction_metrics(measured[holdout], score, classes[holdout],
                                 logits if weight > 0 else None)
    metrics.update(selection_metrics(measured[holdout], score, classes[holdout],
                                     fraction=0.1, ids=ids[holdout]))
    rows.append({'model': name, **metrics})
    del model, optimizer

In [ ]:
table = pd.DataFrame(rows).set_index('model')
columns = ['rmse', 'utility', 'oracle_utility', 'within_type_gap', 'type_allocation_gap']
display(table[columns].round(3))
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
labels = ['MSE', 'MSE + CE', 'SPO+']
axes[0].bar(labels, table.loc[labels, 'rmse'], color=['#476b8a', '#a77550', '#70996a'])
axes[0].set_ylabel('CD16 RMSE (lower is better)')
axes[1].bar(labels, table.loc[labels, 'utility'], color=['#476b8a', '#a77550', '#70996a'])
axes[1].axhline(measured[holdout].mean(), color='0.4', linestyle='--', label='Random selection in expectation')
axes[1].set_ylabel('Top-10% mean CD16 (higher is better)')
axes[1].legend(frameon=False, fontsize=8)
for ax in axes:
    ax.tick_params(axis='x', labelrotation=12)
    ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()
table[columns].to_csv('demo_extended_results.csv')

Selection uses image-based scores. Measured markers and cell types are used for evaluation. The type-allocation and within-type gaps sum to the hindsight gap, which includes unpredictable outcome variation.

SPO+ does not fix a common score offset, so its raw RMSE also reflects calibration.

Sources: [BSCCM](https://doi.org/10.5061/dryad.sxksn038s), [CNN–ViT](https://doi.org/10.1007/978-3-032-31666-0_41), [SPO+](https://doi.org/10.1287/mnsc.2020.3922).
